In [1]:
import pandas as pd
import numpy as np

file_path = "../data/processed/freight_forecasting_dataset.csv"

df = pd.read_csv(file_path)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print(df.head())

Dataset loaded successfully!
Shape: (180, 18)
        Date  Freight_Index  India_Dry_Cargo_Loaded  \
0 2011-01-01           1693               67422.723   
1 2011-02-01           1084               67422.723   
2 2011-03-01           1262               67422.723   
3 2011-04-01           1520               67422.723   
4 2011-05-01           1292               67422.723   

   India_Dry_Cargo_Discharged  Chennai_Total_Traffic  Kamarajar_Total_Traffic  \
0                  127781.219                61057.0                  10703.0   
1                  127781.219                61057.0                  10703.0   
2                  127781.219                61057.0                  10703.0   
3                  127781.219                61460.0                  11009.0   
4                  127781.219                61460.0                  11009.0   

   Paradip_Total_Traffic  SMP_Kolkata_Haldia_Total_Traffic  \
0                57011.0                           46423.0   
1           

In [2]:
# Create freight lag features

df["Freight_Lag_1"] = df["Freight_Index"].shift(1)
df["Freight_Lag_3"] = df["Freight_Index"].shift(3)
df["Freight_Lag_6"] = df["Freight_Index"].shift(6)
df["Freight_Lag_12"] = df["Freight_Index"].shift(12)

print("Freight lag features created!")

print(df[[
    "Date",
    "Freight_Index",
    "Freight_Lag_1",
    "Freight_Lag_3",
    "Freight_Lag_6",
    "Freight_Lag_12"
]].head(15))

Freight lag features created!
         Date  Freight_Index  Freight_Lag_1  Freight_Lag_3  Freight_Lag_6  \
0  2011-01-01           1693            NaN            NaN            NaN   
1  2011-02-01           1084         1693.0            NaN            NaN   
2  2011-03-01           1262         1084.0            NaN            NaN   
3  2011-04-01           1520         1262.0         1693.0            NaN   
4  2011-05-01           1292         1520.0         1084.0            NaN   
5  2011-06-01           1485         1292.0         1262.0            NaN   
6  2011-07-01           1422         1485.0         1520.0         1693.0   
7  2011-08-01           1256         1422.0         1292.0         1084.0   
8  2011-09-01           1682         1256.0         1485.0         1262.0   
9  2011-10-01           1886         1682.0         1422.0         1520.0   
10 2011-11-01           1859         1886.0         1256.0         1292.0   
11 2011-12-01           1862         1859.0   

In [3]:
# Create rolling mean features

df["Freight_Rolling_Mean_3"] = (
    df["Freight_Index"]
    .shift(1)
    .rolling(3)
    .mean()
)

df["Freight_Rolling_Mean_6"] = (
    df["Freight_Index"]
    .shift(1)
    .rolling(6)
    .mean()
)

df["Freight_Rolling_Mean_12"] = (
    df["Freight_Index"]
    .shift(1)
    .rolling(12)
    .mean()
)

print("Rolling mean features created!")

print(df[[
    "Date",
    "Freight_Index",
    "Freight_Rolling_Mean_3",
    "Freight_Rolling_Mean_6",
    "Freight_Rolling_Mean_12"
]].tail(10))

Rolling mean features created!
          Date  Freight_Index  Freight_Rolling_Mean_3  Freight_Rolling_Mean_6  \
170 2025-03-01           1276             1021.666667             1398.666667   
171 2025-04-01           1587             1014.333333             1291.500000   
172 2025-05-01           1411             1200.333333             1217.666667   
173 2025-06-01           1422             1424.666667             1223.166667   
174 2025-07-01           1458             1473.333333             1243.833333   
175 2025-08-01           2018             1430.333333             1315.333333   
176 2025-09-01           2024             1632.666667             1528.666667   
177 2025-10-01           1980             1833.333333             1653.333333   
178 2025-11-01           1945             2007.333333             1718.833333   
179 2025-12-01           2583             1983.000000             1807.833333   

     Freight_Rolling_Mean_12  
170              1635.916667  
171            

In [4]:
# Create 1-month lag features for commodity prices

commodity_cols = [
    "Aluminum",
    "Coal, Australian",
    "Coal, South African **",
    "Copper",
    "Crude oil, Brent",
    "Iron ore, cfr spot",
    "Natural gas, Europe",
    "Soybeans",
    "Wheat, US HRW"
]

for col in commodity_cols:
    df[col + "_Lag_1"] = df[col].shift(1)

print("Commodity lag features created!")

print(df[[
    "Date",
    "Aluminum",
    "Aluminum_Lag_1",
    "Copper",
    "Copper_Lag_1",
    "Crude oil, Brent",
    "Crude oil, Brent_Lag_1"
]].tail())

Commodity lag features created!
          Date  Aluminum  Aluminum_Lag_1   Copper  Copper_Lag_1  \
175 2025-08-01    2597.0          2606.0   9670.0        9771.0   
176 2025-09-01    2653.0          2597.0   9984.0        9670.0   
177 2025-10-01    2793.0          2653.0  10740.0        9984.0   
178 2025-11-01    2819.0          2793.0  10812.0       10740.0   
179 2025-12-01    2876.0          2819.0  11785.0       10812.0   

     Crude oil, Brent  Crude oil, Brent_Lag_1  
175              68.2                    71.0  
176              68.0                    68.2  
177              64.7                    68.0  
178              63.6                    64.7  
179              62.7                    63.6  


In [5]:
# Create 1-month lag features for trade data

trade_cols = [
    "India_Dry_Cargo_Loaded",
    "India_Dry_Cargo_Discharged"
]

for col in trade_cols:
    df[col + "_Lag_1"] = df[col].shift(1)

print("Trade lag features created!")

print(df[[
    "Date",
    "India_Dry_Cargo_Loaded",
    "India_Dry_Cargo_Loaded_Lag_1",
    "India_Dry_Cargo_Discharged",
    "India_Dry_Cargo_Discharged_Lag_1"
]].tail())

Trade lag features created!
          Date  India_Dry_Cargo_Loaded  India_Dry_Cargo_Loaded_Lag_1  \
175 2025-08-01              103525.285                    103525.285   
176 2025-09-01              103525.285                    103525.285   
177 2025-10-01              103525.285                    103525.285   
178 2025-11-01              103525.285                    103525.285   
179 2025-12-01              103525.285                    103525.285   

     India_Dry_Cargo_Discharged  India_Dry_Cargo_Discharged_Lag_1  
175                  368404.571                        368404.571  
176                  368404.571                        368404.571  
177                  368404.571                        368404.571  
178                  368404.571                        368404.571  
179                  368404.571                        368404.571  


In [6]:
# Port traffic features

port_cols = [
    "Chennai_Total_Traffic",
    "Kamarajar_Total_Traffic",
    "Paradip_Total_Traffic",
    "SMP_Kolkata_Haldia_Total_Traffic",
    "Visakhapatnam_Total_Traffic"
]

print("Port features selected:")
print(port_cols)

print("\nMissing values in port features:")
print(df[port_cols].isnull().sum())

Port features selected:
['Chennai_Total_Traffic', 'Kamarajar_Total_Traffic', 'Paradip_Total_Traffic', 'SMP_Kolkata_Haldia_Total_Traffic', 'Visakhapatnam_Total_Traffic']

Missing values in port features:
Chennai_Total_Traffic               57
Kamarajar_Total_Traffic             57
Paradip_Total_Traffic               57
SMP_Kolkata_Haldia_Total_Traffic    57
Visakhapatnam_Total_Traffic         57
dtype: int64


In [7]:
# Create calendar features

df["Month"] = df["Date"].dt.month
df["Quarter"] = df["Date"].dt.quarter
df["Year"] = df["Date"].dt.year

print("Calendar features created!")

print(df[[
    "Date",
    "Month",
    "Quarter",
    "Year"
]].tail())

Calendar features created!
          Date  Month  Quarter  Year
175 2025-08-01      8        3  2025
176 2025-09-01      9        3  2025
177 2025-10-01     10        4  2025
178 2025-11-01     11        4  2025
179 2025-12-01     12        4  2025


In [8]:
# Create next-month forecasting target

df["Target_Freight_Next_Month"] = df["Freight_Index"].shift(-1)

print("Forecast target created!")

print(df[[
    "Date",
    "Freight_Index",
    "Target_Freight_Next_Month"
]].tail(10))

Forecast target created!
          Date  Freight_Index  Target_Freight_Next_Month
170 2025-03-01           1276                     1587.0
171 2025-04-01           1587                     1411.0
172 2025-05-01           1411                     1422.0
173 2025-06-01           1422                     1458.0
174 2025-07-01           1458                     2018.0
175 2025-08-01           2018                     2024.0
176 2025-09-01           2024                     1980.0
177 2025-10-01           1980                     1945.0
178 2025-11-01           1945                     2583.0
179 2025-12-01           2583                        NaN


In [9]:
# Columns that must have values

required_cols = [
    "Freight_Lag_1",
    "Freight_Lag_3",
    "Freight_Lag_6",
    "Freight_Lag_12",
    "Freight_Rolling_Mean_3",
    "Freight_Rolling_Mean_6",
    "Freight_Rolling_Mean_12",
    "Target_Freight_Next_Month"
]

df_features = df.dropna(subset=required_cols).reset_index(drop=True)

print("Feature engineering completed!")
print("Original dataset shape:", df.shape)
print("Feature dataset shape:", df_features.shape)

print("\nRemaining missing values:")
print(df_features.isnull().sum().sum())

Feature engineering completed!
Original dataset shape: (180, 40)
Feature dataset shape: (167, 40)

Remaining missing values:
280


In [10]:
# Save the feature-engineered dataset

output_path = "../data/processed/freight_forecasting_features.csv"

df_features.to_csv(output_path, index=False)

print("Feature dataset saved successfully!")
print("Saved to:", output_path)
print("Final shape:", df_features.shape)

Feature dataset saved successfully!
Saved to: ../data/processed/freight_forecasting_features.csv
Final shape: (167, 40)
